In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

day2_path = "../data/cleaned_data/APL_Day2_Validated.csv"

df_day2 = pd.read_csv(
    day2_path,
    encoding="cp1252"
)

print("Day 2 validated dataset loaded successfully.")
print("Shape:", df_day2.shape)

C:\Users\tejpr\AppData\Local\Temp\ipykernel_11472\3589206240.py:9: DtypeWarning: Columns (0: Customer Zipcode) have mixed types. Specify dtype option on import or set low_memory=False.
  df_day2 = pd.read_csv(


Day 2 validated dataset loaded successfully.
Shape: (180519, 40)


## Working Copy

In [2]:
df_day3 = df_day2.copy()

print("Day 3 working copy created.")
print("Shape:", df_day3.shape)

Day 3 working copy created.
Shape: (180519, 40)


## Feature Group 1 - Delivery Delay

In [3]:
df_day3["Shipping_Delay_Days"] = (
    df_day3["Days for shipping (real)"]
    - df_day3["Days for shipment (scheduled)"]
)   

### Feature: Is_Delayed

In [4]:
df_day3["Is_Delayed"] = (
    df_day3["Shipping_Delay_Days"] > 0
).astype(int)

### Feature: Is_On_Time

In [5]:
df_day3["Is_On_Time"] = (
    df_day3["Shipping_Delay_Days"] == 0
).astype(int)

### Feature: Early_Delivery_Days

In [6]:
df_day3["Early_Delivery_Days"] = (
    df_day3["Days for shipment (scheduled)"]
    - df_day3["Days for shipping (real)"]
).clip(lower=0)

### Feature: Delay_Severity

In [7]:
df_day3["Delay_Severity"] = np.select(
    [
        df_day3["Shipping_Delay_Days"] <= 0,
        df_day3["Shipping_Delay_Days"] == 1,
        df_day3["Shipping_Delay_Days"] >= 2
    ],
    [
        "On Time / Early",
        "Minor Delay",
        "Major Delay"
    ],
    default="Unknown"
)

## Feature Group 2 - Delivery Efficiency

In [8]:
df_day3["Delivery_Efficiency_Ratio"] = np.where(
    df_day3["Days for shipping (real)"] > 0,
    df_day3["Days for shipment (scheduled)"] /
    df_day3["Days for shipping (real)"],
    np.nan
)

## Feature Group 3 - Order Value

### Feature - Revenue_Per_Unit

In [9]:
df_day3["Revenue_Per_Unit"] = np.where(
    df_day3["Order Item Quantity"] > 0,
    df_day3["Sales"] / df_day3["Order Item Quantity"],
    np.nan
)

### Feature - Profit_Margin

In [10]:
df_day3["Profit_Margin"] = np.where(
    df_day3["Sales"] > 0,
    df_day3["Order Profit Per Order"] / df_day3["Sales"],
    np.nan
)

### Feature - Discount_Impact

In [11]:
df_day3["Discount_Level"] = pd.cut(
    df_day3["Order Item Discount Rate"],
    bins=[-np.inf, 0, 0.10, 0.20, np.inf],
    labels=[
        "No Discount",
        "Low Discount",
        "Medium Discount",
        "High Discount"
    ]
)

## Feature Group 4 - Geographic Features

### Feature: Region_Market

In [12]:
df_day3["Region_Market"] = (
    df_day3["Market"].astype(str)
    + " - "
    + df_day3["Order Region"].astype(str)
)

## Feature Group 5 - Shipping Mode Classification  

In [13]:
shipping_mode_rank = {
    "Same Day": 1,
    "First Class": 2,
    "Second Class": 3,
    "Standard Class": 4
}

df_day3["Shipping_Mode_Rank"] = (
    df_day3["Shipping Mode"]
    .map(shipping_mode_rank)
)

## Feature Group 6 - Customer Segment 

In [14]:
df_day3["Is_Corporate_Customer"] = (
    df_day3["Customer Segment"] == "Corporate"
).astype(int)

## Feature Group 7 - Order Quantity Category

In [15]:
df_day3["Order_Quantity_Category"] = pd.cut(
    df_day3["Order Item Quantity"],
    bins=[0, 1, 3, np.inf],
    labels=[
        "Small",
        "Medium",
        "Large"
    ]
)

## Feature Group 8 - Customer/Order Risk Profile

In [16]:
df_day3["Operational_Risk_Profile"] = np.select(
    [
        (df_day3["Late_delivery_risk"] == 1) &
        (df_day3["Shipping Mode"].isin(["Standard Class", "Second Class"])),

        (df_day3["Late_delivery_risk"] == 1),

        (df_day3["Late_delivery_risk"] == 0)
    ],
    [
        "High Risk - Slower Shipping",
        "High Risk",
        "Lower Risk"
    ],
    default="Unknown"
)

## Check the New Features

In [17]:
new_features = [
    "Shipping_Delay_Days",
    "Is_Delayed",
    "Is_On_Time",
    "Early_Delivery_Days",
    "Delay_Severity",
    "Delivery_Efficiency_Ratio",
    "Revenue_Per_Unit",
    "Profit_Margin",
    "Discount_Level",
    "Region_Market",
    "Shipping_Mode_Rank",
    "Is_Corporate_Customer",
    "Order_Quantity_Category",
    "Operational_Risk_Profile"
]

display(df_day3[new_features].head(10))

,Shipping_Delay_Days,Is_Delayed,Is_On_Time,Early_Delivery_Days,Delay_Severity,Delivery_Efficiency_Ratio,Revenue_Per_Unit,Profit_Margin,Discount_Level,Region_Market,Shipping_Mode_Rank,Is_Corporate_Customer,Order_Quantity_Category,Operational_Risk_Profile
0,2,1,0,0,Major Delay,0.666667,99.99,0.319412,Low Discount,Pacific Asia - South Asia,4,0,Large,High Risk - Slower Shipping
1,0,0,1,0,On Time / Early,1.000000,39.99,0.243611,Medium Discount,LATAM - Central America,4,0,Large,Lower Risk
2,0,0,1,0,On Time / Early,1.000000,199.99,0.436822,Low Discount,LATAM - Central America,4,0,Small,Lower Risk
3,2,1,0,0,Major Delay,0.666667,199.99,-0.209460,Medium Discount,USCA - East of USA,4,0,Small,High Risk - Slower Shipping
4,2,1,0,0,Major Delay,0.666667,50.00,0.200000,Medium Discount,USCA - East of USA,4,0,Small,High Risk - Slower Shipping
5,2,1,0,0,Major Delay,0.666667,299.98,0.352123,Low Discount,USCA - East of USA,4,0,Small,High Risk - Slower Shipping
6,0,0,1,0,On Time / Early,1.000000,49.98,0.427504,Low Discount,LATAM - Central America,4,0,Medium,Lower Risk
7,0,0,1,0,On Time / Early,1.000000,49.98,0.226791,Medium Discount,LATAM - Central America,4,0,Medium,Lower Risk
8,0,0,1,0,On Time / Early,1.000000,399.98,0.008800,Medium Discount,Europe - Southern Europe,4,0,Small,Lower Risk
9,0,0,1,0,On Time / Early,1.000000,59.99,0.336223,Medium Discount,LATAM - Central America,4,0,Medium,Lower Risk


## Feature Validation

In [18]:
feature_missing = df_day3[new_features].isnull().sum()

display(
    feature_missing[feature_missing > 0]
)

Delivery_Efficiency_Ratio    5080
dtype: int64

## Validate Delay Feature

In [19]:
delay_check = (
    df_day3["Shipping_Delay_Days"]
    ==
    (
        df_day3["Days for shipping (real)"]
        - df_day3["Days for shipment (scheduled)"]
    )
).all()

print("Shipping delay calculation:", delay_check)

Shipping delay calculation: True


## Validate (Is_Dealy)

In [20]:
delayed_check = (
    df_day3["Is_Delayed"]
    ==
    (df_day3["Shipping_Delay_Days"] > 0).astype(int)
).all()

print("Delayed flag validation:", delayed_check)

Delayed flag validation: True


## Validate Featuer-Engineered Dataset

In [21]:
print("Rows:", df_day3.shape[0])
print("Columns:", df_day3.shape[1])
print("Duplicate rows:", df_day3.duplicated().sum())

Rows: 180519
Columns: 54
Duplicate rows: 0


## Feature Dictionary

In [22]:
feature_dictionary = pd.DataFrame({
    "Feature": [
        "Shipping_Delay_Days",
        "Is_Delayed",
        "Is_On_Time",
        "Early_Delivery_Days",
        "Delay_Severity",
        "Delivery_Efficiency_Ratio",
        "Revenue_Per_Unit",
        "Profit_Margin",
        "Discount_Level",
        "Region_Market",
        "Shipping_Mode_Rank",
        "Is_Corporate_Customer",
        "Order_Quantity_Category",
        "Operational_Risk_Profile"
    ],
    
    "Description": [
        "Difference between actual and scheduled shipping days",
        "Binary indicator for delayed shipment",
        "Binary indicator for exactly on-schedule shipment",
        "Number of days delivered earlier than scheduled",
        "Categorical classification of delivery delay severity",
        "Ratio comparing scheduled shipping time with actual shipping time",
        "Sales generated per ordered unit",
        "Order profit relative to sales",
        "Categorical classification of discount rate",
        "Combined market and order-region feature",
        "Ordinal ranking of shipping service speed",
        "Binary indicator for corporate customers",
        "Categorical classification of order quantity",
        "Combined operational risk classification"
    ],
    
    "Feature_Type": [
        "Numeric",
        "Binary",
        "Binary",
        "Numeric",
        "Categorical",
        "Numeric",
        "Numeric",
        "Numeric",
        "Categorical",
        "Categorical",
        "Ordinal",
        "Binary",
        "Categorical",
        "Categorical"
    ]
})

display(feature_dictionary)

,Feature,Description,Feature_Type
0,Shipping_Delay_Days,Difference between actual and scheduled shippi...,Numeric
1,Is_Delayed,Binary indicator for delayed shipment,Binary
2,Is_On_Time,Binary indicator for exactly on-schedule shipment,Binary
3,Early_Delivery_Days,Number of days delivered earlier than scheduled,Numeric
4,Delay_Severity,Categorical classification of delivery delay s...,Categorical
5,Delivery_Efficiency_Ratio,Ratio comparing scheduled shipping time with a...,Numeric
6,Revenue_Per_Unit,Sales generated per ordered unit,Numeric
7,Profit_Margin,Order profit relative to sales,Numeric
8,Discount_Level,Categorical classification of discount rate,Categorical
9,Region_Market,Combined market and order-region feature,Categorical


## Saving Day 3 Feature Engineering Dataset

In [23]:
import os

os.makedirs("../data/feature_data", exist_ok=True)

In [24]:
day3_path = "../data/feature_data/APL_Day3_Feature_Engineered.csv"

df_day3.to_csv(
    day3_path,
    index=False,
    encoding="cp1252"
)

print("Day 3 feature-engineered dataset saved successfully.")
print(day3_path)

Day 3 feature-engineered dataset saved successfully.
../data/feature_data/APL_Day3_Feature_Engineered.csv


## Save Feature Dictionary

In [25]:
feature_dictionary_path = (
    "../data/feature_data/APL_Day3_Feature_Dictionary.csv"
)

feature_dictionary.to_csv(
    feature_dictionary_path,
    index=False,
    encoding="utf-8"
)

print("Feature dictionary saved successfully.")
print(feature_dictionary_path)

Feature dictionary saved successfully.
../data/feature_data/APL_Day3_Feature_Dictionary.csv


## Final Day 3 Validation

In [26]:
day3_validation = {
    "Row Count Preserved": df_day3.shape[0] == 180519,
    "New Features Created": len(new_features) > 0,
    "Duplicate Rows": df_day3.duplicated().sum() == 0,
    "Delay Feature Correct": delay_check,
    "Delayed Flag Correct": delayed_check,
    "Output File Exists": os.path.exists(day3_path)
}

day3_validation_report = pd.DataFrame(
    list(day3_validation.items()),
    columns=["Validation", "Status"]
)

display(day3_validation_report)

,Validation,Status
0,Row Count Preserved,True
1,New Features Created,True
2,Duplicate Rows,True
3,Delay Feature Correct,True
4,Delayed Flag Correct,True
5,Output File Exists,True


In [27]:
if day3_validation_report["Status"].all():
    print("DAY 3 FEATURE ENGINEERING: PASS")
else:
    print("DAY 3 FEATURE ENGINEERING: CHECK REQUIRED")

DAY 3 FEATURE ENGINEERING: PASS
